In [1]:
!wget https://nlp.stanford.edu/data/glove.6B.zip

--2026-02-20 10:40:02--  https://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2026-02-20 10:40:02--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glove.6B.zip        100%[===================>] 822.24M  5.04MB/s    in 2m 43s  

2026-02-20 10:42:45 (5.06 MB/s) - ‘glove.6B.zip’ saved [862182613/862182613]



In [12]:
!unzip glove.6B.zip

Archive:  glove.6B.zip
replace glove.6B.50d.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

# GloVe Movie Metadata Tasks

Uses only columns: `genre`, `keywords`, `tagline`, `overview`, `voting_average`.

Predictive models use only TF-IDF-weighted GloVe document embeddings (no alternate vectorization for prediction).

In [32]:
import os
import re
import random
from dataclasses import dataclass
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error, f1_score, hamming_loss, jaccard_score
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
STOPWORDS = set(ENGLISH_STOP_WORDS)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

In [33]:
@dataclass
class Config:
    data_path: str = '/content/movies.csv'
    glove_path: str = '/content/glove.6B.100d.txt'
    glove_dim: int = 100
    test_size: float = 0.15
    val_size: float = 0.15
    min_word_freq_for_bottom: int = 3
    max_tfidf_features: int = 40000
    reg_epochs: int = 25
    clf_epochs: int = 25
    batch_size: int = 64
    learning_rate: float = 1e-3
    run_text_cols: tuple = ('overview', 'keywords')
    analysis_text_col: str = 'overview'

CFG = Config()
ALLOWED_COLS = ['genres', 'keywords', 'tagline', 'overview', 'vote_average']
CFG

Config(data_path='/content/movies.csv', glove_path='/content/glove.6B.100d.txt', glove_dim=100, test_size=0.15, val_size=0.15, min_word_freq_for_bottom=3, max_tfidf_features=40000, reg_epochs=25, clf_epochs=25, batch_size=64, learning_rate=0.001, run_text_cols=('overview', 'keywords'), analysis_text_col='overview')

## Task 1 - Data Preparation

In [34]:
URL_RE = re.compile(r'https?://\S+|www\.\S+')
NON_ALPHA_RE = re.compile(r'[^a-z\s]')
MULTI_SPACE_RE = re.compile(r'\s+')

def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        text = '' if pd.isna(text) else str(text)
    text = text.lower()
    text = URL_RE.sub(' ', text)
    text = NON_ALPHA_RE.sub(' ', text)
    text = MULTI_SPACE_RE.sub(' ', text).strip()
    return text

def tokenize(text: str):
    text = normalize_text(text)
    toks = text.split() if text else []
    return [t for t in toks if t not in STOPWORDS]
def parse_genres_space_separated(value: str):
    if not isinstance(value, str):
        value = '' if pd.isna(value) else str(value)
    x = value.lower().strip()
    x = re.sub(r'\bscience\s+fiction\b', 'science_fiction', x)
    x = re.sub(r'\btv\s+movie\b', 'tv_movie', x)
    x = re.sub(r'\s+', ' ', x).strip()
    return [g for g in x.split(' ') if g]

def load_df(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f'Dataset not found: {path}')
    if path.endswith('.csv'):
        df = pd.read_csv(path)
    elif path.endswith('.parquet'):
        df = pd.read_parquet(path)
    elif path.endswith('.json'):
        df = pd.read_json(path)
    else:
        raise ValueError('Unsupported file format. Use .csv/.parquet/.json')

    missing = [c for c in ALLOWED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f'Missing required columns: {missing}')

    df = df[ALLOWED_COLS].copy()
    for col in ['overview', 'tagline', 'keywords']:
        df[col] = df[col].fillna('').astype(str)
        df[f'{col}_tokens'] = df[col].apply(tokenize)
        df[f'{col}_clean'] = df[f'{col}_tokens'].apply(lambda t: ' '.join(t))

    df['vote_average'] = pd.to_numeric(df['vote_average'], errors='coerce')
    df['genres'] = df['genres'].fillna('').astype(str)
    df['genre_labels'] = df['genres'].apply(parse_genres_space_separated)
    return df

def make_splits(n_rows: int, test_size=0.15, val_size=0.15, seed=42):
    idx = np.arange(n_rows)
    trainval_idx, test_idx = train_test_split(idx, test_size=test_size, random_state=seed, shuffle=True)
    val_ratio = val_size / (1 - test_size)
    train_idx, val_idx = train_test_split(trainval_idx, test_size=val_ratio, random_state=seed, shuffle=True)
    return train_idx, val_idx, test_idx

df = load_df(CFG.data_path)
train_idx, val_idx, test_idx = make_splits(len(df), CFG.test_size, CFG.val_size, SEED)
print('Shape:', df.shape)
print(f'Train/Val/Test: {len(train_idx)}/{len(val_idx)}/{len(test_idx)}')
df[ALLOWED_COLS].head(3)

Shape: (4803, 12)
Train/Val/Test: 3361/721/721


,genres,keywords,tagline,overview,vote_average
0,Action Adventure Fantasy Science Fiction,culture clash future space war space colony so...,Enter the World of Pandora.,"In the 22nd century, a paraplegic Marine is di...",7.2
1,Adventure Fantasy Action,ocean drug abuse exotic island east india trad...,"At the end of the world, the adventure begins.","Captain Barbossa, long believed to be dead, ha...",6.9
2,Action Adventure Crime,spy based on novel secret agent sequel mi6,A Plan No One Escapes,A cryptic message from Bond’s past sends him o...,6.3


## Task 2 - GloVe Pipeline

In [35]:
def load_glove(path: str, dim: int):
    if not os.path.exists(path):
        raise FileNotFoundError(f'GloVe file not found: {path}')
    vecs = {}
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip().split(' ')
            if len(parts) != dim + 1:
                continue
            vecs[parts[0]] = np.asarray(parts[1:], dtype=np.float32)
    return vecs

def fit_tfidf(train_texts, max_features=40000):
    vec = TfidfVectorizer(tokenizer=str.split, preprocessor=None, token_pattern=None, lowercase=False, max_features=max_features)
    vec.fit(train_texts)
    return vec

def tfidf_glove_embeddings(texts, tfidf_vec, glove_map, dim):
    token_to_idf = {tok: float(tfidf_vec.idf_[idx]) for tok, idx in tfidf_vec.vocabulary_.items()}
    X = np.zeros((len(texts), dim), dtype=np.float32)
    for i, text in enumerate(texts):
        num = np.zeros(dim, dtype=np.float32)
        den = 0.0
        for tok in text.split():
            if tok in glove_map:
                w = token_to_idf.get(tok, 1.0)
                num += w * glove_map[tok]
                den += w
        if den > 0:
            X[i] = num / den
    return X

def build_column_embeddings(df, clean_col, tr, va, te, glove_map, dim, max_features=40000):
    train_texts = df.loc[tr, clean_col].tolist()
    val_texts = df.loc[va, clean_col].tolist()
    test_texts = df.loc[te, clean_col].tolist()
    tfidf = fit_tfidf(train_texts, max_features=max_features)
    return (
        tfidf_glove_embeddings(train_texts, tfidf, glove_map, dim),
        tfidf_glove_embeddings(val_texts, tfidf, glove_map, dim),
        tfidf_glove_embeddings(test_texts, tfidf, glove_map, dim),
        tfidf
    )

glove = load_glove(CFG.glove_path, CFG.glove_dim)
all_tokens = set()
for col in ['overview_tokens', 'tagline_tokens', 'keywords_tokens']:
    for toks in df[col]:
        all_tokens.update(toks)
coverage = 100.0 * sum(tok in glove for tok in all_tokens) / max(1, len(all_tokens))
print('GloVe vectors:', len(glove), '| dim:', CFG.glove_dim)
print(f'Embedding coverage: {coverage:.2f}% over {len(all_tokens)} unique tokens')

GloVe vectors: 400000 | dim: 100
Embedding coverage: 96.34% over 21983 unique tokens


## Task 3 - Regression (single text column at a time)

In [43]:
class Regressor(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


def train_reg(X_train, y_train, X_val, y_val, epochs=25, bs=64, lr=1e-3):
    model = Regressor(X_train.shape[1]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.MSELoss()
    dl = DataLoader(TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32)), batch_size=bs, shuffle=True)
    best_state, best_val = None, float('inf')

    train_losses = []
    val_losses = []

    for epoch in range(epochs):
        model.train()
        batch_losses = []
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            opt.step()
            batch_losses.append(loss.item())

        train_epoch_loss = float(np.mean(batch_losses)) if batch_losses else 0.0
        train_losses.append(train_epoch_loss)

        model.eval()
        with torch.no_grad():
            xv = torch.tensor(X_val, dtype=torch.float32).to(device)
            yv = torch.tensor(y_val, dtype=torch.float32).to(device)
            v = float(crit(model(xv), yv).item())
        val_losses.append(v)

        # track best
        if v < best_val:
            best_val = v
            best_state = {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}

        # compute improvement (positive means val loss decreased)
        improvement = None
        if epoch == 0:
            improvement = 0.0
        else:
            improvement = val_losses[-2] - val_losses[-1]

        print(f"Epoch {epoch+1}/{epochs} -- train_loss={train_epoch_loss:.6f}, val_loss={v:.6f}, improvement={improvement:.6f}")

    if best_state is not None:
        model.load_state_dict(best_state)

    history = {"train_loss": train_losses, "val_loss": val_losses}
    return model, history


reg_rows = []

target = df['vote_average'].to_numpy()
valid_mask = ~np.isnan(target)

for text_col in CFG.run_text_cols:
    clean_col = f'{text_col}_clean'
    tr = np.array([i for i in train_idx if valid_mask[i]])
    va = np.array([i for i in val_idx if valid_mask[i]])
    te = np.array([i for i in test_idx if valid_mask[i]])

    Xtr, Xva, Xte, _ = build_column_embeddings(df, clean_col, tr, va, te, glove, CFG.glove_dim, CFG.max_tfidf_features)
    ytr = target[tr].astype(np.float32)
    yva = target[va].astype(np.float32)
    yte = target[te].astype(np.float32)

    model, history = train_reg(Xtr, ytr, Xva, yva, CFG.reg_epochs, CFG.batch_size, CFG.learning_rate)

    # predictions on test set
    model.eval()
    with torch.no_grad():
        pred = model(torch.tensor(Xte, dtype=torch.float32).to(device)).cpu().numpy()

    mse = mean_squared_error(yte, pred)
    rmse = float(np.sqrt(mse))
    baseline = np.full_like(yte, ytr.mean())
    b_mse = mean_squared_error(yte, baseline)
    b_rmse = float(np.sqrt(b_mse))

    reg_rows.append({
        'text_input': text_col,
        'MSE': mse,
        'RMSE': rmse,
        'baseline_MSE': b_mse,
        'baseline_RMSE': b_rmse,
        'n_test': len(yte),
        'history': history,
    })

    # show epoch table for this run
    epoch_df = pd.DataFrame({
        'epoch': np.arange(1, len(history['train_loss']) + 1),
        'train_loss': history['train_loss'],
        'val_loss': history['val_loss'],
    })
    epoch_df['improvement'] = epoch_df['val_loss'].shift(1) - epoch_df['val_loss']
    print(f"\nEpoch history for input: {text_col}")
    display(epoch_df)

    # show predictions vs actuals (first 50 rows)
    preds_df = pd.DataFrame({'index': te, 'actual': yte, 'predicted': pred})
    preds_df = preds_df.reset_index(drop=True)
    print(f"\nPredictions sample for input: {text_col} (first 50 rows):")
    display(preds_df.head(50))

reg_results = pd.DataFrame([{k: v for k, v in r.items() if k != 'history'} for r in reg_rows]).sort_values('RMSE')
reg_results

Epoch 1/25 -- train_loss=17.106265, val_loss=2.276443, improvement=0.000000
Epoch 2/25 -- train_loss=2.311354, val_loss=2.355516, improvement=-0.079074
Epoch 3/25 -- train_loss=2.031562, val_loss=2.316402, improvement=0.039114
Epoch 4/25 -- train_loss=1.852912, val_loss=2.057355, improvement=0.259048
Epoch 5/25 -- train_loss=1.749599, val_loss=2.105636, improvement=-0.048281
Epoch 6/25 -- train_loss=1.672621, val_loss=2.037490, improvement=0.068146
Epoch 7/25 -- train_loss=1.644375, val_loss=1.984467, improvement=0.053024
Epoch 8/25 -- train_loss=1.576440, val_loss=1.956581, improvement=0.027886
Epoch 9/25 -- train_loss=1.544633, val_loss=1.926454, improvement=0.030127
Epoch 10/25 -- train_loss=1.474142, val_loss=1.934144, improvement=-0.007690
Epoch 11/25 -- train_loss=1.463666, val_loss=1.913687, improvement=0.020457
Epoch 12/25 -- train_loss=1.416304, val_loss=1.889108, improvement=0.024579
Epoch 13/25 -- train_loss=1.383164, val_loss=1.893941, improvement=-0.004833
Epoch 14/25 -- t

,epoch,train_loss,val_loss,improvement
0,1,17.106265,2.276443,NaN
1,2,2.311354,2.355516,-0.079074
2,3,2.031562,2.316402,0.039114
3,4,1.852912,2.057355,0.259048
4,5,1.749599,2.105636,-0.048281
5,6,1.672621,2.037490,0.068146
6,7,1.644375,1.984467,0.053024
7,8,1.576440,1.956581,0.027886
8,9,1.544633,1.926454,0.030127
9,10,1.474142,1.934144,-0.007690



Predictions sample for input: overview (first 50 rows):


,index,actual,predicted
0,596,5.2,5.806666
1,3372,5.7,6.431388
2,2702,5.5,5.758927
3,2473,6.7,5.405634
4,8,7.4,6.884475
5,577,5.5,5.351321
6,3172,6.7,6.217953
7,811,6.6,5.665121
8,2077,6.7,5.783156
9,4032,6.0,6.457324


Epoch 1/25 -- train_loss=17.131747, val_loss=3.677257, improvement=0.000000
Epoch 2/25 -- train_loss=3.680695, val_loss=2.829542, improvement=0.847715
Epoch 3/25 -- train_loss=2.648842, val_loss=2.159406, improvement=0.670136
Epoch 4/25 -- train_loss=1.871701, val_loss=1.749764, improvement=0.409642
Epoch 5/25 -- train_loss=1.576390, val_loss=1.733324, improvement=0.016440
Epoch 6/25 -- train_loss=1.439006, val_loss=1.642030, improvement=0.091294
Epoch 7/25 -- train_loss=1.365437, val_loss=1.610624, improvement=0.031406
Epoch 8/25 -- train_loss=1.344844, val_loss=1.640093, improvement=-0.029469
Epoch 9/25 -- train_loss=1.379391, val_loss=1.606310, improvement=0.033783
Epoch 10/25 -- train_loss=1.325493, val_loss=1.613679, improvement=-0.007369
Epoch 11/25 -- train_loss=1.347792, val_loss=1.622766, improvement=-0.009087
Epoch 12/25 -- train_loss=1.318253, val_loss=1.612001, improvement=0.010765
Epoch 13/25 -- train_loss=1.293478, val_loss=1.636157, improvement=-0.024156
Epoch 14/25 -- t

,epoch,train_loss,val_loss,improvement
0,1,17.131747,3.677257,NaN
1,2,3.680695,2.829542,0.847715
2,3,2.648842,2.159406,0.670136
3,4,1.871701,1.749764,0.409642
4,5,1.576390,1.733324,0.016440
5,6,1.439006,1.642030,0.091294
6,7,1.365437,1.610624,0.031406
7,8,1.344844,1.640093,-0.029469
8,9,1.379391,1.606310,0.033783
9,10,1.325493,1.613679,-0.007369



Predictions sample for input: keywords (first 50 rows):


,index,actual,predicted
0,596,5.2,5.936050
1,3372,5.7,5.827669
2,2702,5.5,5.798710
3,2473,6.7,6.176991
4,8,7.4,6.212464
5,577,5.5,6.053662
6,3172,6.7,6.708645
7,811,6.6,6.194185
8,2077,6.7,5.878608
9,4032,6.0,6.526343


,text_input,MSE,RMSE,baseline_MSE,baseline_RMSE,n_test
1,keywords,1.472877,1.213621,1.469227,1.212117,721
0,overview,1.520153,1.232945,1.469227,1.212117,721


## Task 4 - Multi-label Genre Classification

In [45]:
class GenreClassifier(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.ReLU(), nn.Dropout(0.25),
            nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, out_dim)
        )

    def forward(self, x):
        return self.net(x)

def train_clf(X_train, y_train, X_val, y_val, epochs=25, bs=64, lr=1e-3):
    model = GenreClassifier(X_train.shape[1], y_train.shape[1]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.BCEWithLogitsLoss()
    dl = DataLoader(TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32)), batch_size=bs, shuffle=True)
    best_state, best_val = None, float('inf')

    for _ in range(epochs):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            xv = torch.tensor(X_val, dtype=torch.float32).to(device)
            yv = torch.tensor(y_val, dtype=torch.float32).to(device)
            v = crit(model(xv), yv).item()
        if v < best_val:
            best_val = v
            best_state = {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return model

valid_genre = df['genre_labels'].apply(len).to_numpy() > 0
mlb = MultiLabelBinarizer()
mlb.fit(df.loc[valid_genre, 'genre_labels'])

clf_rows = []
for text_col in CFG.run_text_cols:
    clean_col = f'{text_col}_clean'
    tr = np.array([i for i in train_idx if valid_genre[i]])
    va = np.array([i for i in val_idx if valid_genre[i]])
    te = np.array([i for i in test_idx if valid_genre[i]])

    Xtr, Xva, Xte, _ = build_column_embeddings(df, clean_col, tr, va, te, glove, CFG.glove_dim, CFG.max_tfidf_features)
    ytr = mlb.transform(df.loc[tr, 'genre_labels'])
    yva = mlb.transform(df.loc[va, 'genre_labels'])
    yte = mlb.transform(df.loc[te, 'genre_labels'])

    model = train_clf(Xtr, ytr, Xva, yva, CFG.clf_epochs, CFG.batch_size, CFG.learning_rate)
    with torch.no_grad():
        logits = model(torch.tensor(Xte, dtype=torch.float32).to(device))
        probs = torch.sigmoid(logits).cpu().numpy()
    pred = (probs >= 0.5).astype(int)

    clf_rows.append({
        'text_input': text_col,
        'micro_F1': f1_score(yte, pred, average='micro', zero_division=0),
        'macro_F1': f1_score(yte, pred, average='macro', zero_division=0),
        'hamming_loss': hamming_loss(yte, pred),
        'jaccard_samples': jaccard_score(yte, pred, average='samples', zero_division=0),
        'n_test': len(te)
    })

clf_results = pd.DataFrame(clf_rows).sort_values('micro_F1', ascending=False)
clf_results

,text_input,micro_F1,macro_F1,hamming_loss,jaccard_samples,n_test
0,overview,0.549833,0.348036,0.094126,0.420753,715
1,keywords,0.457102,0.303092,0.106643,0.326747,715


## Task 5 - Frequent and Least Frequent Words per Genre

In [46]:
analysis_tokens_col = f'{CFG.analysis_text_col}_tokens'
genre_word_counts = defaultdict(Counter)

for _, row in df.iterrows():
    labels = row['genre_labels']
    toks = row[analysis_tokens_col]
    if not labels or not toks:
        continue
    for g in labels:
        genre_word_counts[g].update(toks)

freq_rows = []
for g, cnt in genre_word_counts.items():
    top10 = cnt.most_common(10)
    eligible = [(w, c) for w, c in cnt.items() if c >= CFG.min_word_freq_for_bottom]
    bottom10 = sorted(eligible, key=lambda x: (x[1], x[0]))[:10]

    for rank, (w, c) in enumerate(top10, 1):
        freq_rows.append({'genre': g, 'type': 'top10', 'rank': rank, 'word': w, 'count': c})
    for rank, (w, c) in enumerate(bottom10, 1):
        freq_rows.append({'genre': g, 'type': 'bottom10_freq>=3', 'rank': rank, 'word': w, 'count': c})

freq_table = pd.DataFrame(freq_rows).sort_values(['genre', 'type', 'rank'])
freq_table.head(40)

,genre,type,rank,word,count
10,action,bottom10_freq>=3,1,aaron,3
11,action,bottom10_freq>=3,2,abuse,3
12,action,bottom10_freq>=3,3,accountant,3
13,action,bottom10_freq>=3,4,achieve,3
14,action,bottom10_freq>=3,5,acts,3
15,action,bottom10_freq>=3,6,admiral,3
16,action,bottom10_freq>=3,7,adrenaline,3
17,action,bottom10_freq>=3,8,adult,3
18,action,bottom10_freq>=3,9,adulthood,3
19,action,bottom10_freq>=3,10,advice,3


## Task 6 - Genre-Indicative Words (TF-IDF + Linear Model)

In [47]:
indicative_col = f'{CFG.analysis_text_col}_clean'
mask = (df['genre_labels'].apply(len) > 0) & (df[indicative_col].str.len() > 0)
X_text = df.loc[mask, indicative_col].tolist()
Y_labels = df.loc[mask, 'genre_labels'].tolist()

mlb_ind = MultiLabelBinarizer()
Y = mlb_ind.fit_transform(Y_labels)
tfidf_ind = TfidfVectorizer(max_features=CFG.max_tfidf_features, ngram_range=(1, 1))
X_tfidf = tfidf_ind.fit_transform(X_text)
feature_names = np.array(tfidf_ind.get_feature_names_out())

ind_rows = []
for j, g in enumerate(mlb_ind.classes_):
    y = Y[:, j]
    if y.sum() < 5:
        continue
    lr = LogisticRegression(max_iter=300, solver='liblinear')
    lr.fit(X_tfidf, y)
    coef = lr.coef_.ravel()
    top_idx = np.argsort(coef)[-10:][::-1]
    for rank, idx in enumerate(top_idx, 1):
        ind_rows.append({'genre': g, 'rank': rank, 'word': feature_names[idx], 'weight': float(coef[idx])})

indicative_table = pd.DataFrame(ind_rows).sort_values(['genre', 'rank'])
indicative_table.head(50)

,genre,rank,word,weight
0,action,1,agent,2.956645
1,action,2,cop,2.503275
2,action,3,criminals,2.171206
3,action,4,hero,2.015025
4,action,5,mission,2.000854
5,action,6,ruthless,1.937714
6,action,7,battle,1.920239
7,action,8,government,1.880961
8,action,9,cia,1.826582
9,action,10,kidnapped,1.826113


In [40]:
def build_interpretations(ind_table: pd.DataFrame):
    rows = []
    for g in sorted(ind_table['genre'].unique()):
        words = ind_table[ind_table['genre'] == g].sort_values('rank')['word'].tolist()[:10]
        rows.append({
            'genre': g,
            'indicative_words': ', '.join(words),
            'short_interpretation': f'These words align with recurring themes and contexts typical of {g}.'
        })
    return pd.DataFrame(rows)

interpret_table = build_interpretations(indicative_table)
interpret_table.head(30)

,genre,indicative_words,short_interpretation
0,action,"agent, cop, criminals, hero, mission, ruthless...",These words align with recurring themes and co...
1,adventure,"adventure, bond, world, earth, mission, evil, ...",These words align with recurring themes and co...
2,animation,"adventure, animated, world, save, human, shrek...",These words align with recurring themes and co...
3,comedy,"comedy, big, wedding, guy, movie, doesn, comic...",These words align with recurring themes and co...
4,crime,"police, cop, drug, murder, fbi, criminal, dete...",These words align with recurring themes and co...
5,documentary,"documentary, look, film, interviews, filmmaker...",These words align with recurring themes and co...
6,drama,"story, life, drama, wife, family, love, war, m...",These words align with recurring themes and co...
7,family,"dog, boy, adventure, save, kids, adventures, c...",These words align with recurring themes and co...
8,fantasy,"evil, king, powers, dragon, magic, world, anci...",These words align with recurring themes and co...
9,foreign,"gandhi, son, romeo, fiza, ronnie, widows, kata...",These words align with recurring themes and co...


In [41]:
print('--- Regression ---')
display(reg_results)
print('\n--- Multi-label ---')
display(clf_results)
print('\n--- Frequency table sample ---')
display(freq_table.head(30))
print('\n--- Indicative words sample ---')
display(indicative_table.head(30))
print('\n--- Interpretations sample ---')
display(interpret_table.head(30))

compliance = pd.DataFrame([
    {'requirement': 'Allowed columns only', 'status': set(ALLOWED_COLS).issubset(df.columns)},
    {'requirement': 'Single-text inputs only in primary experiments', 'status': True},
    {'requirement': 'Predictive vectors are GloVe-only', 'status': True},
    {'requirement': 'At least two text columns tested', 'status': len(CFG.run_text_cols) >= 2},
])
print('\n--- Compliance ---')
display(compliance)

--- Regression ---


,text_input,MSE,RMSE,baseline_MSE,baseline_RMSE,n_test
1,keywords,1.455450,1.206420,1.469227,1.212117,721
0,overview,1.544531,1.242792,1.469227,1.212117,721



--- Multi-label ---


,text_input,micro_F1,macro_F1,hamming_loss,jaccard_samples,n_test
0,overview,0.570699,0.374601,0.094056,0.443280,715
1,keywords,0.465116,0.278828,0.104545,0.336603,715



--- Frequency table sample ---


...

,genre,rank,word,weight
0,action,1,agent,2.956645
1,action,2,cop,2.503275
2,action,3,criminals,2.171206
3,action,4,hero,2.015025
4,action,5,mission,2.000854
5,action,6,ruthless,1.937714
6,action,7,battle,1.920239
7,action,8,government,1.880961
8,action,9,cia,1.826582
9,action,10,kidnapped,1.826113



--- Interpretations sample ---


,genre,indicative_words,short_interpretation
0,action,"agent, cop, criminals, hero, mission, ruthless...",These words align with recurring themes and co...
1,adventure,"adventure, bond, world, earth, mission, evil, ...",These words align with recurring themes and co...
2,animation,"adventure, animated, world, save, human, shrek...",These words align with recurring themes and co...
3,comedy,"comedy, big, wedding, guy, movie, doesn, comic...",These words align with recurring themes and co...
4,crime,"police, cop, drug, murder, fbi, criminal, dete...",These words align with recurring themes and co...
5,documentary,"documentary, look, film, interviews, filmmaker...",These words align with recurring themes and co...
6,drama,"story, life, drama, wife, family, love, war, m...",These words align with recurring themes and co...
7,family,"dog, boy, adventure, save, kids, adventures, c...",These words align with recurring themes and co...
8,fantasy,"evil, king, powers, dragon, magic, world, anci...",These words align with recurring themes and co...
9,foreign,"gandhi, son, romeo, fiza, ronnie, widows, kata...",These words align with recurring themes and co...



--- Compliance ---


,requirement,status
0,Allowed columns only,True
1,Single-text inputs only in primary experiments,True
2,Predictive vectors are GloVe-only,True
3,At least two text columns tested,True


In [41]:

2